## Minimal MS MARCO Learning-to-Rank (L2R) with Elasticsearch + XGBoost

This notebook is a minimal working prototype:

- BM25 retrieval via Elasticsearch (candidates)
- Basic feature extraction
- Train a Learning-to-Rank model (`xgboost.XGBRanker`)
- Rerank the top results
- Evaluate with NDCG@10

Assumptions:
- Elasticsearch index: `msmarco` with fields `pid`, `text`
- Local files: `queries.tsv`, `qrels.train.tsv`
- ES config in `.env.local`: `ES_LOCAL_URL`, `ES_LOCAL_API_KEY`

## Step 1: Setup and Imports

Import libraries, load `.env.local`, initialize Elasticsearch client, and define helper functions.

In [2]:
import os
import re
import random
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from elasticsearch import Elasticsearch
from sklearn.model_selection import train_test_split  # imported per requirements (not used)
from xgboost import XGBRanker


def load_dotenv_like(path: str) -> Dict[str, str]:
    """Minimal .env loader (supports ${VAR} expansion)."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing env file: {path}")

    env = dict(os.environ)
    var_ref = re.compile(r"\$\{([A-Za-z_][A-Za-z0-9_]*)\}")

    def expand(value: str) -> str:
        def repl(m):
            k = m.group(1)
            return env.get(k, os.environ.get(k, ""))
        return var_ref.sub(repl, value)

    with open(path, "r", encoding="utf-8") as f:
        for raw in f:
            line = raw.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            k, v = line.split("=", 1)
            k = k.strip()
            v = v.strip().strip('"').strip("'")
            env[k] = expand(v)
    return env


ENV = load_dotenv_like(".env.local")

ES_URL = ENV.get("ES_LOCAL_URL")
ES_API_KEY = ENV.get("ES_LOCAL_API_KEY")
ES_INDEX = ENV.get("ES_INDEX", "msmarco")

if not ES_URL:
    raise ValueError("ES_LOCAL_URL not found in .env.local")
if not ES_API_KEY:
    raise ValueError("ES_LOCAL_API_KEY not found in .env.local")

es = Elasticsearch(ES_URL, api_key=ES_API_KEY)


def tokenize(text: str) -> List[str]:
    return str(text).lower().split()


def snippet(text: str, n: int = 180) -> str:
    s = re.sub(r"\s+", " ", str(text)).strip()
    return s[:n] + ("…" if len(s) > n else "")


def bm25_search(query: str, k: int = 10, index: str = None) -> List[Dict]:
    index = index or ES_INDEX
    q = {"match": {"passage": query}}
    try:
        resp = es.search(index=index, query=q, size=k)
    except TypeError:
        # fallback for older client versions
        resp = es.search(index=index, body={"query": q}, size=k)

    hits = resp.get("hits", {}).get("hits", [])
    out = []
    for h in hits:
        src = h.get("_source", {})
        pid = src.get("pid", h.get("_id"))
        out.append(
            {
                "pid": str(pid),
                "passage": src.get("passage", ""),
                "bm25_score": float(h.get("_score", 0.0)),
            }
        )
    return out


print({"ES_URL": ES_URL, "ES_INDEX": ES_INDEX, "api_key_loaded": bool(ES_API_KEY)})
# 1) Ping + basic info
print("Ping:", es.ping())
print("Info:", es.info()["version"]["number"])

# 2) Index exists?
exists = es.indices.exists(index=ES_INDEX)
print("Index exists:", exists)

# 3) Count docs (quick check that data is there)
if exists:
    try:
        cnt = es.count(index=ES_INDEX)["count"]
        print("Doc count:", cnt)
    except Exception as e:
        print("Count failed:", e)


{'ES_URL': 'http://localhost:9200', 'ES_INDEX': 'msmarco', 'api_key_loaded': True}
Ping: True
Info: 9.3.2
Index exists: True
Doc count: 8841823


## Step 2: Load Data

Load `queries.tsv` and `qrels.train.tsv` and show basic stats.

In [3]:
queries = pd.read_csv(
    "queries.train.tsv",
    sep="\t",
    names=["qid", "query"],
    dtype={"qid": str, "query": str},
)

qrels = pd.read_csv(
    "qrels.train.tsv",
    sep="\t",
    names=["qid", "unused", "pid", "relevance"],
    dtype={"qid": str, "pid": str, "relevance": int},
)
qrels = qrels.drop(columns=["unused"])

display(queries.head())
print("#queries rows:", len(queries), "unique qids:", queries["qid"].nunique())
print("#qrels rows:", len(qrels), "unique qids:", qrels["qid"].nunique())
print("qrels relevance distribution:")
print(qrels["relevance"].value_counts().sort_index())

,qid,query
0,121352,define extreme
1,634306,what does chattel mean on credit history
2,920825,what was the great leap forward brainly
3,510633,tattoo fixers how much does it cost
4,737889,what is decentralization process.


#queries rows: 808731 unique qids: 808731
#qrels rows: 532761 unique qids: 502939
qrels relevance distribution:
relevance
1    532761
Name: count, dtype: int64


## Step 3: Sample Queries

Sample ~50 queries that have labels (appear in qrels).

In [15]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

qids_with_labels = set(qrels["qid"].unique())
queries_labeled = queries[queries["qid"].isin(qids_with_labels)].copy()

sample_size = min(10000, len(queries_labeled))
sampled_queries = queries_labeled.sample(n=sample_size, random_state=SEED).reset_index(drop=True)

display(sampled_queries.head(10))
print("sampled qids:", sampled_queries["qid"].nunique())

,qid,query
0,593879,what causes your heart rate to slow down
1,956520,when was abraham lincoln born and killed
2,108293,cost per pothole to fix
3,597677,what color is argan oil
4,606166,"what county is fredericksburg, va?"
5,683326,what is a first bachelor degree
6,1163095,what district is mill creek in indiana
7,503802,stimulus package definition
8,1044203,who makes parts for electric cars
9,223404,how do you play the card game full deck


sampled qids: 10000


## Step 4: BM25 Retrieval Demo (visual)

For 3–5 queries, show BM25 top-10 (pid + snippet + score).

In [16]:
demo_n = min(5, len(sampled_queries))
demo_rows = sampled_queries.sample(n=demo_n, random_state=SEED)

for _, row in demo_rows.iterrows():
    qid, qtext = row["qid"], row["query"]
    print("\n" + "=" * 80)
    print(f"QID {qid}: {qtext}")
    hits = bm25_search(qtext, k=10)
    for rank, h in enumerate(hits, start=1):
        print(f"  {rank:2d}. pid={h['pid']}  bm25={h['bm25_score']:.4f}  text=" + snippet(h["passage"]))


QID 319764: how much for a replacement window
   1. pid=6241282  bm25=28.1532  text=How much do Pella replacement windows cost? Pella replacement windows have a wide range of costs depending on the size and type of window. Expect to pay between $75 and $1,500 per …
   2. pid=1317190  bm25=26.5208  text=Get an estimate of how much new windows will cost using this window replacement cost calculator. Quick and easy to use, receive a quote within minutes. Get an estimate of how much …
   3. pid=3629184  bm25=24.4534  text=How Much Do Home Replacement Windows Cost. According to REMODELING Magazineâs 2015 Window Cost vs. Value Report, a mid-range vinyl window replacement will cost you $11,198, based…
   4. pid=1411798  bm25=24.4008  text=How Much Do Home Replacement Windows Cost. According to REMODELING Magazineâs 2015 Window Cost vs. Value Report, a mid-range vinyl window replacement will cost you $11,198, based…
   5. pid=810718  bm25=24.1363  text=Comparison Of Window Prices Per Proj

## Step 5: Build Candidate Set

For each sampled query, retrieve top-100 BM25 docs and store `(qid, pid, bm25_score, text)`.

In [17]:
candidate_k = 100
candidate_rows = []

for _, row in sampled_queries.iterrows():
    qid, qtext = row["qid"], row["query"]
    hits = bm25_search(qtext, k=candidate_k)
    for h in hits:
        candidate_rows.append(
            {
                "qid": qid,
                "query": qtext,
                "pid": str(h["pid"]),
                "bm25_score": float(h["bm25_score"]),
                "passage": h["passage"],
            }
        )

candidates = pd.DataFrame(candidate_rows)
print(
    "candidates rows:",
    len(candidates),
    "unique qids:",
    candidates["qid"].nunique(),
    "unique pids:",
    candidates["pid"].nunique(),
)
display(candidates.head())

candidates rows: 999920 unique qids: 10000 unique pids: 810440


,qid,query,pid,bm25_score,passage
0,593879,what causes your heart rate to slow down,4683391,35.413280,What Causes Bradycardia. Your heart has a buil...
1,593879,what causes your heart rate to slow down,1328257,33.942013,Symptom Checker. 1 Slow heart rate and Cardio...
2,593879,what causes your heart rate to slow down,5897267,33.587017,Causes of Slow heart rate that are very common...
3,593879,what causes your heart rate to slow down,1328252,32.907795,Classifications of Slow heart rate: Medical Co...
4,593879,what causes your heart rate to slow down,6634662,32.827232,How to Slow Down Your Heart Rate Naturally. Ra...


## Step 6: Label Construction

Label = 1 if `(qid, pid)` is relevant (`relevance > 0`), else 0. Show class balance.

In [18]:
qrels_sampled = qrels[qrels["qid"].isin(sampled_queries["qid"])].copy()

relevant_pairs = set(
    zip(
        qrels_sampled.loc[qrels_sampled["relevance"] > 0, "qid"],
        qrels_sampled.loc[qrels_sampled["relevance"] > 0, "pid"],
    )
)

candidates["label"] = [
    1 if (qid, pid) in relevant_pairs else 0
    for qid, pid in zip(candidates["qid"], candidates["pid"])
]

print("label balance:")
print(candidates["label"].value_counts())

label balance:
label
0    993696
1      6224
Name: count, dtype: int64


## Step 7: Feature Engineering (basic)

Features per (query, doc):
- BM25 score
- Term overlap count
- Query coverage (fraction of query terms present)
- Document length

In [19]:
# --- Precompute query tokens ---
qid_to_qtokens = {
    qid: tokenize(qtext)
    for qid, qtext in sampled_queries[["qid", "query"]].itertuples(index=False, name=None)
}

# --- Helpers ---
def bigrams(tokens):
    return list(zip(tokens, tokens[1:]))

def min_pairwise_distance(pos_a, pos_b):
    i = j = 0
    best = None
    while i < len(pos_a) and j < len(pos_b):
        a, b = pos_a[i], pos_b[j]
        d = abs(a - b)
        best = d if best is None else min(best, d)
        if a < b:
            i += 1
        else:
            j += 1
    return best

# --- Feature function ---
def featurize_row(qid: str, doc_text: str, bm25_score: float):
    q_tokens = qid_to_qtokens.get(qid, [])
    d_tokens = tokenize(doc_text)
    d_set = set(d_tokens)
    d_text = " ".join(d_tokens)

    # --- Basic features ---
    overlap = sum(1 for t in q_tokens if t in d_set)
    coverage = overlap / max(1, len(q_tokens))
    doc_len = len(d_tokens)

    # --- Phrase match ---
    q_phrase = " ".join(q_tokens)
    phrase_match = int(q_phrase in d_text) if q_phrase else 0

    # --- Proximity (FIXED: use inverse, not raw distance) ---
    q_unique = list(dict.fromkeys(q_tokens))
    if len(q_unique) < 2:
        prox_inv = 0.0
    else:
        positions = {}
        for i, tok in enumerate(d_tokens):
            if tok in q_unique:
                positions.setdefault(tok, []).append(i)

        if any(t not in positions for t in q_unique):
            prox_inv = 0.0
        else:
            best = None
            for i in range(len(q_unique)):
                for j in range(i + 1, len(q_unique)):
                    d = min_pairwise_distance(
                        positions[q_unique[i]],
                        positions[q_unique[j]],
                    )
                    best = d if best is None else min(best, d)

            prox_inv = 1.0 / (1.0 + best) if best is not None else 0.0

    # --- Bigram match (FIXED: normalized) ---
    q_bigrams = bigrams(q_tokens)
    if q_bigrams:
        d_bigrams_set = set(bigrams(d_tokens))
        matches = sum(1 for bg in q_bigrams if bg in d_bigrams_set)
        bigram_score = matches / len(q_bigrams)
    else:
        bigram_score = 0.0

    return (
        float(bm25_score),
        float(overlap),
        float(coverage),
        float(doc_len),
        float(phrase_match),
        float(prox_inv),
        float(bigram_score),
    )

# --- Feature columns ---
feat_cols = [
    "bm25_score",
    "term_overlap_count",
    "query_coverage",
    "doc_length",
    "phrase_match",
    "proximity_inv",
    "bigram_score",
]

# --- Compute features ---
features = np.vstack(
    [
        featurize_row(qid, txt, score)
        for qid, txt, score in zip(
            candidates["qid"],
            candidates["passage"],
            candidates["bm25_score"],
        )
    ]
)

for i, c in enumerate(feat_cols):
    candidates[c] = features[:, i]

# --- Preview ---
display(candidates[["qid", "pid"] + feat_cols + ["label"]].head())

,qid,pid,bm25_score,term_overlap_count,query_coverage,doc_length,phrase_match,proximity_inv,bigram_score,label
0,593879,4683391,35.413280,7.0,0.875,39.0,0.0,0.0,0.285714,0
1,593879,1328257,33.942013,3.0,0.375,54.0,0.0,0.0,0.142857,0
2,593879,5897267,33.587017,4.0,0.500,92.0,0.0,0.0,0.142857,0
3,593879,1328252,32.907795,3.0,0.375,31.0,0.0,0.0,0.142857,0
4,593879,6634662,32.827232,6.0,0.750,64.0,0.0,0.0,0.571429,0


## Step 8: Prepare Training Data

Sort by `qid` and build `X`, `y`, and `group` (docs per query).

In [20]:
candidates_sorted = candidates.sort_values(["qid", "bm25_score"], ascending=[True, False]).reset_index(drop=True)

X = candidates_sorted[feat_cols].to_numpy(dtype=np.float32)
y = candidates_sorted["label"].to_numpy(dtype=np.float32)

group = candidates_sorted.groupby("qid").size().to_list()

print(
    "X shape:", X.shape,
    "y shape:", y.shape,
    "#groups:", len(group),
    "min/max group size:", min(group), max(group),
)

X shape: (999920, 7) y shape: (999920,) #groups: 10000 min/max group size: 20 100


## Step 9: Train L2R Model

Train an `XGBRanker` with a fast configuration.

In [21]:
ranker = XGBRanker(
    objective="rank:ndcg",
    n_estimators=60,
    learning_rate=0.1,
    max_depth=4,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=SEED,
    n_jobs=4,
)

ranker.fit(X, y, group=group)

print("Trained XGBRanker on", len(group), "queries and", X.shape[0], "(query,doc) pairs")

try:
    fi = ranker.feature_importances_
    for name, val in sorted(zip(feat_cols, fi), key=lambda x: -x[1]):
        print(f"  {name}: {val:.4f}")
except Exception as e:
    print("(feature importance unavailable)", e)

Trained XGBRanker on 10000 queries and 999920 (query,doc) pairs
  bm25_score: 0.5402
  query_coverage: 0.1357
  proximity_inv: 0.1178
  bigram_score: 0.0683
  term_overlap_count: 0.0626
  phrase_match: 0.0501
  doc_length: 0.0254


## Step 10: Reranking Demo

For a few queries, compare BM25 top-10 vs L2R reranked top-10.

In [22]:
rerank_demo_n = min(3, sampled_queries["qid"].nunique())
demo_qids = sampled_queries["qid"].drop_duplicates().sample(n=rerank_demo_n, random_state=SEED).to_list()

for qid in demo_qids:
    qtext = sampled_queries.loc[sampled_queries["qid"] == qid, "query"].iloc[0]
    dfq = candidates_sorted[candidates_sorted["qid"] == qid].head(10).copy()

    Xq = dfq[feat_cols].to_numpy(dtype=np.float32)
    dfq["pred_score"] = ranker.predict(Xq)

    bm25_view = dfq[["pid", "bm25_score", "pred_score", "label", "passage"]].copy()
    bm25_view["snippet"] = bm25_view["passage"].map(snippet)
    bm25_view = bm25_view.drop(columns=["passage"]).reset_index(drop=True)

    reranked_view = bm25_view.sort_values("pred_score", ascending=False).reset_index(drop=True)

    print("\n" + "=" * 80)
    print(f"QID {qid}: {qtext}")
    print("\nBM25 top-10:")
    display(bm25_view[["pid", "bm25_score", "pred_score", "label", "snippet"]])
    print("\nL2R reranked top-10:")
    display(reranked_view[["pid", "bm25_score", "pred_score", "label", "snippet"]])


QID 319764: how much for a replacement window

BM25 top-10:


,pid,bm25_score,pred_score,label,snippet
0,6241282,28.153168,0.686463,0,How much do Pella replacement windows cost? Pe...
1,1317190,26.520754,0.399257,0,Get an estimate of how much new windows will c...
2,3629184,24.453426,0.259443,0,How Much Do Home Replacement Windows Cost. Acc...
3,1411798,24.400843,0.245323,0,How Much Do Home Replacement Windows Cost. Acc...
4,810718,24.136263,0.618698,0,Comparison Of Window Prices Per Project. When ...
5,6241285,23.929953,0.245323,0,How Much Do Home Replacement Windows Cost Plus...
6,4756387,23.220110,0.192059,0,How Much Do Home Replacement Windows Cost Plus...
7,2701329,23.218527,0.566449,0,How much it costs to replace your windows depe...
8,5836379,23.182903,0.566449,0,One thing you should look out for when getting...
9,3171270,23.162294,0.566449,0,For an idea of how much a door replacement can...



L2R reranked top-10:


,pid,bm25_score,pred_score,label,snippet
0,6241282,28.153168,0.686463,0,How much do Pella replacement windows cost? Pe...
1,810718,24.136263,0.618698,0,Comparison Of Window Prices Per Project. When ...
2,5836379,23.182903,0.566449,0,One thing you should look out for when getting...
3,3171270,23.162294,0.566449,0,For an idea of how much a door replacement can...
4,2701329,23.218527,0.566449,0,How much it costs to replace your windows depe...
5,1317190,26.520754,0.399257,0,Get an estimate of how much new windows will c...
6,3629184,24.453426,0.259443,0,How Much Do Home Replacement Windows Cost. Acc...
7,1411798,24.400843,0.245323,0,How Much Do Home Replacement Windows Cost. Acc...
8,6241285,23.929953,0.245323,0,How Much Do Home Replacement Windows Cost Plus...
9,4756387,23.220110,0.192059,0,How Much Do Home Replacement Windows Cost Plus...



QID 1162576: what does a gastric sleeve mean to me

BM25 top-10:


...

,pid,bm25_score,pred_score,label,snippet
0,2545740,31.650988,0.768258,0,A Lighter Me works with some of the most exper...
1,6557045,29.204115,0.603072,0,How much does gastric sleeve surgery cost? The...
2,7135417,30.021000,0.582602,0,What does the phrase wear (one's) heart on (on...
3,2545737,29.573133,0.556742,0,A Lighter Me works with some of the most exper...
4,2270071,29.392912,0.556742,0,Recovery after Gastric Sleeve Surgery. Day One...
5,266729,28.439420,0.530922,0,"On January 1st, 2010 United Healthcare added g..."
6,5894785,28.583101,0.521796,0,What is a Gastric Sleeve? Surgeons perform lap...
7,3744098,28.146402,0.521796,0,What to Expect After Gastric Sleeve Surgery. A...
8,346985,28.360195,0.492170,0,Of the weight loss surgeries currently availab...
9,8297217,28.263103,0.492170,0,Gastric Sleeve vs. Gastric Bypass: A Compariso...



QID 443386: lpn oklahoma how long is program

BM25 top-10:


,pid,bm25_score,pred_score,label,snippet
0,5895210,32.350384,0.812213,1,The approved LPN programs in Oklahoma only tak...
1,5523758,27.669897,0.551001,0,One Year LPN Training Program â In order to ...
2,3365113,26.627245,0.330233,0,How long does it take for an LPN to become an ...
3,5895214,25.829048,0.301962,0,LPN Degree and Education in Oklahoma. In the s...
4,6282263,25.526447,0.295014,0,How Long Does an LPN to BSN Program Take to Co...
5,5175037,25.297632,0.367395,0,How Long is a LPN / LVN Program. Choosing to e...
6,3054309,25.113350,0.018415,0,3 Licensed Practical Nurse LPN / LVN Salary in...
7,6375433,25.011179,0.265918,0,How long will it take to finish an LPN program...
8,2546263,24.030994,0.092980,0,According to payscale.com LPNs make an average...
9,5895212,23.238651,-0.024581,0,Oklahoma has a growing number of opportunities...



L2R reranked top-10:


,pid,bm25_score,pred_score,label,snippet
0,5895210,32.350384,0.812213,1,The approved LPN programs in Oklahoma only tak...
1,5523758,27.669897,0.551001,0,One Year LPN Training Program â In order to ...
2,5175037,25.297632,0.367395,0,How Long is a LPN / LVN Program. Choosing to e...
3,3365113,26.627245,0.330233,0,How long does it take for an LPN to become an ...
4,5895214,25.829048,0.301962,0,LPN Degree and Education in Oklahoma. In the s...
5,6282263,25.526447,0.295014,0,How Long Does an LPN to BSN Program Take to Co...
6,6375433,25.011179,0.265918,0,How long will it take to finish an LPN program...
7,2546263,24.030994,0.092980,0,According to payscale.com LPNs make an average...
8,3054309,25.113350,0.018415,0,3 Licensed Practical Nurse LPN / LVN Salary in...
9,5895212,23.238651,-0.024581,0,Oklahoma has a growing number of opportunities...


## Step 11: Basic Evaluation (NDCG@10)

Compute mean NDCG@10 for BM25 vs L2R rerank over the sampled queries.

In [23]:
def dcg_at_k(rels: List[float], k: int = 10) -> float:
    rels = rels[:k]
    return float(sum((2**r - 1) / np.log2(i + 2) for i, r in enumerate(rels))) if rels else 0.0

def ndcg_at_k(rels: List[float], k: int = 10) -> float:
    dcg = dcg_at_k(rels, k=k)
    ideal = dcg_at_k(sorted(rels, reverse=True), k=k)
    return 0.0 if ideal == 0 else float(dcg / ideal)

bm25_scores = []
l2r_scores = []

for qid, dfq in candidates_sorted.groupby("qid"):
    top = dfq.head(10).copy()

    rels_bm25 = top["label"].astype(float).to_list()

    Xq = top[feat_cols].to_numpy(dtype=np.float32)
    preds = ranker.predict(Xq)
    rels_l2r = top.assign(pred=preds).sort_values("pred", ascending=False)["label"].astype(float).to_list()

    bm25_scores.append(ndcg_at_k(rels_bm25, k=10))
    l2r_scores.append(ndcg_at_k(rels_l2r, k=10))

print(f"Mean NDCG@10 (BM25): {np.mean(bm25_scores):.4f}")
print(f"Mean NDCG@10 (L2R rerank): {np.mean(l2r_scores):.4f}")

Mean NDCG@10 (BM25): 0.1969
Mean NDCG@10 (L2R rerank): 0.2038


Looking in indexes: https://download.pytorch.org/whl/cpu
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.3/190.3 MB 13.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 kB 29.3 MB/s eta 0:00:00


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")